# 🚚 FreightQuote AI - Milestone 3 Consolidated Enterprise Notebook
This notebook combines all features from Milestone 1 and Milestone 2 into a single, clean, modular, and production-ready Google Colab deployment.

### Notebook Architecture
- **db_engine.py**: SQLite schema, JWT tokens, validations, progressive account lockout, and OTP cooldown.
- **ml_engine.py**: Generates synthetic shipping data and trains 5 regression/classification algorithms per agent, selecting the best model.
- **copilot_engine.py**: Loads Qwen2.5-3B-Instruct in 4-bit NF4 quantization on GPU, with high-fidelity CPU rule-based fallback.
- **app.py**: A complete, secure, responsive Streamlit dashboard containing calculators, admin user manager, and the AI copilot chat interface.

### Prerequisites
To deploy the Streamlit frontend with ngrok, click the **Secrets** tab (key icon) on the left sidebar in Google Colab, and add:
1. `NGROK_AUTH_TOKEN`: Your ngrok tunnel credentials.
2. `GMAIL_OTP_EMAIL` (Optional): Gmail SMTP sender email.
3. `GMAIL_OTP_PASSWORD` (Optional): Gmail SMTP App Password.

*(If SMTP credentials are not set, OTP codes will log to `mock_otp.txt` and display as dynamic notifications in the login screen for testing convenience.)*

## 📦 Step 1: Install Enterprise Dependencies
Installs UI libraries, machine learning algorithms, and HuggingFace acceleration utilities.

In [ ]:
# Install pip packages
!pip install -q streamlit pyjwt bcrypt pyngrok scikit-learn joblib pandas numpy matplotlib seaborn
# Quantization dependencies
!pip install -q transformers bitsandbytes accelerate torch

## 🔑 Step 2: Set Up Environment & Colab Secrets
Validates if the ngrok token is loaded properly from Google Colab Secrets.

In [ ]:
import os
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
    print("✅ NGROK_AUTH_TOKEN secret successfully accessed.")
except Exception as e:
    print(f"⚠️ Google Colab secrets not detected or token missing: {e}")
    print("Make sure NGROK_AUTH_TOKEN is defined in the Secrets panel.")

## 🔒 Step 3: Write Database & Security Engine (`db_engine.py`)
Handles SQLite DB setups, bcrypt password hashing, JWT authorization tokens, password strength checks, progressive lockout rules, and OTP cooldown states.

In [ ]:
%%writefile db_engine.py
import sqlite3
import bcrypt
import jwt
import re
import time
import datetime
import random
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from typing import Tuple, Dict, Optional

DB_FILE = "freightquote.db"
JWT_SECRET = "freightquote_jwt_secret_key_12345"

def init_db(db_path: str = DB_FILE):
    """Initializes the SQLite database with the necessary tables."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Create Users table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        security_question TEXT NOT NULL,
        security_answer TEXT NOT NULL,
        role TEXT NOT NULL DEFAULT 'User',
        is_locked INTEGER NOT NULL DEFAULT 0,
        lock_until INTEGER NOT NULL DEFAULT 0,
        failed_attempts INTEGER NOT NULL DEFAULT 0,
        created_at TEXT NOT NULL
    )
    """)
    
    # Create OTP table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS otps (
        email TEXT PRIMARY KEY,
        otp_code TEXT NOT NULL,
        expires_at INTEGER NOT NULL,
        cooldown_level INTEGER NOT NULL DEFAULT 0,
        last_requested_at INTEGER NOT NULL DEFAULT 0
    )
    """)
    
    # Create default Admin if not exists
    cursor.execute("SELECT * FROM users WHERE email = 'admin@freightquote.ai'")
    if not cursor.fetchone():
        hashed = bcrypt.hashpw("Admin@1234".encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
        cursor.execute("""
        INSERT INTO users (email, password_hash, security_question, security_answer, role, created_at)
        VALUES ('admin@freightquote.ai', ?, 'What is your favorite color?', 'Blue', 'Admin', ?)
        """, (hashed, datetime.datetime.now().isoformat()))
        
    conn.commit()
    conn.close()

# Password & Email Validations
def validate_email(email: str) -> bool:
    pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"
    return bool(re.match(pattern, email))

def validate_password(password: str) -> bool:
    if len(password) < 8:
        return False
    if not re.search(r"[a-z]", password):
        return False
    if not re.search(r"[A-Z]", password):
        return False
    if not re.search(r"[0-9]", password):
        return False
    if not re.search(r"[!@#$%^&*(),.?\":{}|<>]", password):
        return False
    return True

def check_password_strength(password: str) -> str:
    score = 0
    if len(password) >= 8:
        score += 1
    if re.search(r"[a-z]", password) and re.search(r"[A-Z]", password):
        score += 1
    if re.search(r"[0-9]", password):
        score += 1
    if re.search(r"[!@#$%^&*(),.?\":{}|<>]", password):
        score += 1
        
    if score <= 2:
        return "Weak"
    elif score == 3:
        return "Average"
    else:
        return "Good"

# JWT Tokens
def create_jwt_token(email: str, role: str) -> str:
    payload = {
        "email": email,
        "role": role,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def decode_jwt_token(token: str) -> Optional[Dict]:
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except (jwt.ExpiredSignatureError, jwt.InvalidTokenError):
        return None

# Lockout Management
def check_user_lockout(email: str, db_path: str = DB_FILE) -> Tuple[bool, str]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT is_locked, lock_until, failed_attempts FROM users WHERE email = ?", (email,))
    row = cursor.fetchone()
    conn.close()
    
    if not row:
        return False, "User not found"
        
    is_locked, lock_until, failed_attempts = row
    current_time = int(time.time())
    
    if is_locked:
        if lock_until == 9999999999:
            return True, "Account is permanently locked. Contact an admin to unlock."
        elif current_time < lock_until:
            remaining = lock_until - current_time
            minutes = remaining // 60
            seconds = remaining % 60
            return True, f"Account locked. Try again in {minutes}m {seconds}s."
        else:
            # Lock expired, reset
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("UPDATE users SET is_locked = 0, lock_until = 0 WHERE email = ?", (email,))
            conn.commit()
            conn.close()
            
    return False, ""

def handle_failed_login(email: str, db_path: str = DB_FILE) -> str:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT failed_attempts FROM users WHERE email = ?", (email,))
    row = cursor.fetchone()
    if not row:
        conn.close()
        return "User not found"
        
    attempts = row[0] + 1
    current_time = int(time.time())
    lock_until = 0
    is_locked = 0
    msg = ""
    
    if attempts == 3:
        lock_until = current_time + 300  # 5 min
        is_locked = 1
        msg = "Account locked for 5 minutes due to 3 failed attempts."
    elif attempts == 4:
        lock_until = current_time + 900  # 15 min
        is_locked = 1
        msg = "Account locked for 15 minutes due to 4 failed attempts."
    elif attempts >= 5:
        lock_until = 9999999999  # Permanent
        is_locked = 1
        msg = "Account permanently locked due to 5+ failed attempts. Contact an admin."
    else:
        msg = f"Failed attempt. {3 - attempts} attempts remaining before lockout."
        
    cursor.execute("""
    UPDATE users 
    SET failed_attempts = ?, is_locked = ?, lock_until = ? 
    WHERE email = ?
    """, (attempts, is_locked, lock_until, email))
    conn.commit()
    conn.close()
    return msg

def reset_failed_login(email: str, db_path: str = DB_FILE):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("UPDATE users SET failed_attempts = 0, is_locked = 0, lock_until = 0 WHERE email = ?", (email,))
    conn.commit()
    conn.close()

# OTP Management
def request_otp(email: str, db_path: str = DB_FILE) -> Tuple[bool, str, str]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT cooldown_level, last_requested_at FROM otps WHERE email = ?", (email,))
    row = cursor.fetchone()
    
    current_time = int(time.time())
    cooldowns = [60, 180, 300, 3600] # levels 0, 1, 2, 3+
    
    if row:
        cooldown_level, last_requested_at = row
        level = min(cooldown_level, 3)
        cooldown_duration = cooldowns[level]
        elapsed = current_time - last_requested_at
        
        if elapsed < cooldown_duration:
            conn.close()
            remaining = cooldown_duration - elapsed
            return False, f"Please wait {remaining} seconds before requesting a new OTP.", ""
            
        next_level = min(cooldown_level + 1, 3)
    else:
        next_level = 0
        
    # Generate 6-digit OTP
    otp_code = f"{random.randint(100000, 999999)}"
    expires_at = current_time + 300 # 5 min expiry
    
    cursor.execute("""
    INSERT OR REPLACE INTO otps (email, otp_code, expires_at, cooldown_level, last_requested_at)
    VALUES (?, ?, ?, ?, ?)
    """, (email, otp_code, expires_at, next_level, current_time))
    
    conn.commit()
    conn.close()
    
    return True, f"OTP generated successfully.", otp_code

def send_otp_email(email: str, otp_code: str) -> bool:
    try:
        from google.colab import userdata
        sender_email = userdata.get('GMAIL_OTP_EMAIL')
        sender_password = userdata.get('GMAIL_OTP_PASSWORD')
    except Exception:
        sender_email = None
        sender_password = None
        
    # Always log to local file for validation convenience
    with open("mock_otp.txt", "w") as f:
        f.write(f"{email}:{otp_code}")
        
    if not sender_email or not sender_password:
        print(f"[OTP LOG - LOCAL FALLBACK] OTP code for {email} is: {otp_code}")
        return True
        
    try:
        msg = MIMEMultipart()
        msg['From'] = sender_email
        msg['To'] = email
        msg['Subject'] = "FreightQuote AI - OTP Verification"
        body = f"Your verification code is: {otp_code}. This code is valid for 5 minutes."
        msg.attach(MIMEText(body, 'plain'))
        
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, email, msg.as_string())
        server.quit()
        return True
    except Exception as e:
        print(f"[OTP ERROR] SMTP failed: {e}. Logged to mock_otp.txt instead.")
        return True

def verify_otp(email: str, code: str, db_path: str = DB_FILE) -> Tuple[bool, str]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT otp_code, expires_at FROM otps WHERE email = ?", (email,))
    row = cursor.fetchone()
    conn.close()
    
    if not row:
        return False, "No OTP requested for this email."
        
    otp_code, expires_at = row
    current_time = int(time.time())
    
    if current_time > expires_at:
        return False, "OTP has expired. Please request a new one."
        
    if otp_code != code:
        return False, "Invalid OTP code."
        
    return True, "OTP verified successfully."


## 📊 Step 4: Write Machine Learning Pipeline (`ml_engine.py`)
Generates comprehensive synthetic shipping data, trains 5 regressors/classifiers per agent, evaluates validation metrics, choosing and saving the best model using Joblib.

In [ ]:
%%writefile ml_engine.py
import json
import joblib
import pandas as pd
import numpy as np
from typing import Dict, Any

# Scikit-learn models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

# Regression algorithms
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Classification algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC

# Set random seed for reproducibility
np.random.seed(42)

def generate_pricing_data(n_samples: int = 1200) -> pd.DataFrame:
    """Generates synthetic freight pricing data."""
    distance = np.random.uniform(50, 3000, n_samples)
    weight = np.random.uniform(1000, 45000, n_samples)
    fuel_index = np.random.uniform(2.5, 5.0, n_samples)
    rating = np.random.uniform(1.0, 5.0, n_samples)
    
    # Target pricing function with complexity and noise
    price = (120 
             + (distance * 1.65) 
             + (weight * 0.045) 
             + (fuel_index * 115) 
             - (rating * 35) 
             + np.random.normal(0, 120, n_samples))
    price = np.clip(price, 120, None)
    
    return pd.DataFrame({
        'distance_miles': distance,
        'weight_lbs': weight,
        'fuel_price_index': fuel_index,
        'carrier_rating': rating,
        'price_usd': price
    })

def generate_route_data(n_samples: int = 1200) -> pd.DataFrame:
    """Generates synthetic route delay prediction data."""
    distance = np.random.uniform(50, 3000, n_samples)
    traffic = np.random.uniform(0.1, 1.0, n_samples)
    weather = np.random.uniform(0.1, 1.0, n_samples)
    carrier_delay_rate = np.random.uniform(0.05, 0.45, n_samples)
    
    # Non-linear probability of delay
    prob = 0.08 + (distance / 3500) * 0.18 + (traffic ** 1.5) * 0.32 + (weather ** 1.2) * 0.28 + carrier_delay_rate * 0.12
    prob = np.clip(prob, 0.02, 0.98)
    is_delayed = np.random.binomial(1, prob)
    
    return pd.DataFrame({
        'distance_miles': distance,
        'traffic_density': traffic,
        'weather_severity': weather,
        'carrier_delay_rate': carrier_delay_rate,
        'is_delayed': is_delayed
    })

def generate_compliance_data(n_samples: int = 1200) -> pd.DataFrame:
    """Generates synthetic carrier compliance classification data."""
    safety_score = np.random.uniform(45, 100, n_samples)
    insurance = np.random.binomial(1, 0.94, n_samples) # 94% have valid insurance
    maintenance = np.random.binomial(1, 0.88, n_samples) # 88% pass safety inspections
    violations = np.random.poisson(0.6, n_samples)
    years = np.random.uniform(1, 20, n_samples)
    
    prob = np.zeros(n_samples)
    for i in range(n_samples):
        if insurance[i] == 0:
            # High probability of non-compliance if no insurance
            prob[i] = 0.02
        else:
            p = 0.35 + (safety_score[i] / 100) * 0.35 + maintenance[i] * 0.20 - min(violations[i] * 0.18, 0.45) + (years[i] / 20) * 0.10
            prob[i] = np.clip(p, 0.01, 0.99)
            
    is_compliant = np.random.binomial(1, prob)
    
    return pd.DataFrame({
        'safety_score': safety_score,
        'insurance_validity': insurance,
        'maintenance_checks_pass': maintenance,
        'violations_count': violations,
        'years_in_service': years,
        'is_compliant': is_compliant
    })

def train_pricing_models() -> Dict[str, Any]:
    df = generate_pricing_data()
    X = df.drop(columns=['price_usd'])
    y = df['price_usd']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train 5 different regression models
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(alpha=1.0),
        'Lasso': Lasso(alpha=0.1),
        'Decision Tree': DecisionTreeRegressor(max_depth=6, random_state=42),
        'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
    }
    
    results = {}
    best_model_name = None
    best_r2 = -float('inf')
    best_model = None
    
    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        
        r2 = r2_score(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        
        results[name] = {'R2': float(r2), 'RMSE': float(rmse), 'MAE': float(mae)}
        
        if r2 > best_r2:
            best_r2 = r2
            best_model_name = name
            best_model = model
            
    # Save best models
    joblib.dump(scaler, "pricing_scaler.joblib")
    joblib.dump(best_model, "best_pricing_model.joblib")
    
    with open("pricing_metrics.json", "w") as f:
        json.dump({"best_model": best_model_name, "metrics": results}, f, indent=4)
        
    return {"best": best_model_name, "metrics": results}

def train_route_delay_models() -> Dict[str, Any]:
    df = generate_route_data()
    X = df.drop(columns=['is_delayed'])
    y = df['is_delayed']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train 5 classification models
    models = {
        'Logistic Regression': LogisticRegression(),
        'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
        'SVC': SVC(probability=True, random_state=42)
    }
    
    results = {}
    best_model_name = None
    best_f1 = -float('inf')
    best_model = None
    
    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        
        try:
            probs = model.predict_proba(X_test_scaled)[:, 1]
            auc = roc_auc_score(y_test, probs)
        except Exception:
            auc = 0.5
            
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        f1 = f1_score(y_test, preds, zero_division=0)
        
        results[name] = {
            'Accuracy': float(acc),
            'ROC-AUC': float(auc),
            'Precision': float(prec),
            'Recall': float(rec),
            'F1': float(f1)
        }
        
        if f1 > best_f1:
            best_f1 = f1
            best_model_name = name
            best_model = model
            
    joblib.dump(scaler, "route_scaler.joblib")
    joblib.dump(best_model, "best_route_delay_model.joblib")
    
    with open("route_metrics.json", "w") as f:
        json.dump({"best_model": best_model_name, "metrics": results}, f, indent=4)
        
    return {"best": best_model_name, "metrics": results}

def train_carrier_compliance_models() -> Dict[str, Any]:
    df = generate_compliance_data()
    X = df.drop(columns=['is_compliant'])
    y = df['is_compliant']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train 5 classification models
    models = {
        'Logistic Regression': LogisticRegression(),
        'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
        'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42)
    }
    
    results = {}
    best_model_name = None
    best_f1 = -float('inf')
    best_model = None
    
    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        
        try:
            probs = model.predict_proba(X_test_scaled)[:, 1]
            auc = roc_auc_score(y_test, probs)
        except Exception:
            auc = 0.5
            
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        f1 = f1_score(y_test, preds, zero_division=0)
        
        results[name] = {
            'Accuracy': float(acc),
            'ROC-AUC': float(auc),
            'Precision': float(prec),
            'Recall': float(rec),
            'F1': float(f1)
        }
        
        if f1 > best_f1:
            best_f1 = f1
            best_model_name = name
            best_model = model
            
    joblib.dump(scaler, "compliance_scaler.joblib")
    joblib.dump(best_model, "best_carrier_compliance_model.joblib")
    
    with open("compliance_metrics.json", "w") as f:
        json.dump({"best_model": best_model_name, "metrics": results}, f, indent=4)
        
    return {"best": best_model_name, "metrics": results}

def run_ml_pipeline():
    """Runs training on all 3 agents and prints select best model metrics."""
    print("Training Agent 1: Dynamic Freight Pricing...")
    pricing = train_pricing_models()
    print("Training Agent 2: Route Delay Prediction...")
    route = train_route_delay_models()
    print("Training Agent 3: Carrier Compliance Classifier...")
    compliance = train_carrier_compliance_models()
    
    # Write Model Card Markdown file
    model_card_md = f"""# FreightQuote AI - Machine Learning Model Card

This model card details the execution results, dataset parameters, and selected models for FreightQuote AI.

## Agent 1: Dynamic Freight Pricing (Regression)
- **Objective**: Predict price_usd for a given shipment.
- **Winner Algorithm**: **{pricing['best']}**
- **Validation Metrics**:
  - R²: {pricing['metrics'][pricing['best']]['R2']:.4f}
  - RMSE: ${pricing['metrics'][pricing['best']]['RMSE']:.2f}
  - MAE: ${pricing['metrics'][pricing['best']]['MAE']:.2f}

## Agent 2: Route Delay Prediction (Classification)
- **Objective**: Classify whether a scheduled route will suffer from transit delays.
- **Winner Algorithm**: **{route['best']}**
- **Validation Metrics**:
  - Accuracy: {route['metrics'][route['best']]['Accuracy']:.4f}
  - ROC-AUC: {route['metrics'][route['best']]['ROC-AUC']:.4f}
  - Precision: {route['metrics'][route['best']]['Precision']:.4f}
  - Recall: {route['metrics'][route['best']]['Recall']:.4f}
  - F1-Score: {route['metrics'][route['best']]['F1']:.4f}

## Agent 3: Carrier Compliance Prediction (Classification)
- **Objective**: Check if a carrier complies with regulatory guidelines (safety score, insurance validity, etc.).
- **Winner Algorithm**: **{compliance['best']}**
- **Validation Metrics**:
  - Accuracy: {compliance['metrics'][compliance['best']]['Accuracy']:.4f}
  - ROC-AUC: {compliance['metrics'][compliance['best']]['ROC-AUC']:.4f}
  - Precision: {compliance['metrics'][compliance['best']]['Precision']:.4f}
  - Recall: {compliance['metrics'][compliance['best']]['Recall']:.4f}
  - F1-Score: {compliance['metrics'][compliance['best']]['F1']:.4f}
"""
    with open("ml_model_card.md", "w") as f:
        f.write(model_card_md)
        
    print("\nML Model Pipeline Executed & Model Card Written.")


## 🤖 Step 5: Write AI Copilot Engine (`copilot_engine.py`)
Loads Qwen2.5-3B-Instruct in 4-bit Nf4. Utilizes a bulletproof rule-based fallback if GPU is unavailable or memory constraints are met.

In [ ]:
%%writefile copilot_engine.py
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class AICopilot:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.gpu_available = torch.cuda.is_available()
        
    def load_model(self) -> bool:
        """Loads Qwen2.5-3B-Instruct in 4-bit quantization if GPU is available."""
        model_id = "Qwen/Qwen2.5-3B-Instruct"
        print(f"[AICopilot] GPU Availability: {self.gpu_available}")
        
        if not self.gpu_available:
            print("[AICopilot] GPU not found. Falling back to high-fidelity rule-based recommendation generator.")
            return False
            
        try:
            print(f"[AICopilot] Attempting to load {model_id} in 4-bit NF4...")
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16
            )
            
            # Using low CPU memory load as well
            self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True
            )
            print("[AICopilot] GPU 4-bit NF4 load succeeded.")
            return True
        except Exception as e:
            print(f"[AICopilot] GPU load failed: {e}. Falling back to rule-based generator.")
            self.model = None
            self.tokenizer = None
            return False

    def generate_recommendation(self, distance: float, weight: float, traffic: float, weather: float, safety_score: float) -> str:
        """Generates shipping/routing carrier recommendations in JSON format."""
        
        prompt_text = f"""
        Analyze logistics metrics and generate a route carrier selection recommendation.
        
        Input Details:
        - Transit Distance: {distance:.1f} miles
        - Cargo Weight: {weight:.1f} lbs
        - Traffic Congestion Level: {traffic:.2f} (0=empty, 1=blocked)
        - Extreme Weather Index: {weather:.2f} (0=clear, 1=severe storm)
        - Carrier Safety Score: {safety_score:.1f} (out of 100)
        
        Generate a structured JSON output with the exact keys:
        - "price_status": "Fair" / "High" / "Low"
        - "delay_risk": "Low" / "Medium" / "High"
        - "compliance_risk": "Low" / "Medium" / "High"
        - "recommendation_text": "A descriptive, professional 1-2 sentence assessment."
        """
        
        if self.model is not None and self.tokenizer is not None:
            try:
                messages = [
                    {"role": "system", "content": "You are FreightQuote AI Copilot. Return ONLY a single JSON block."},
                    {"role": "user", "content": prompt_text}
                ]
                # Format using chat template
                text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                model_inputs = self.tokenizer([text], return_tensors="pt").to("cuda")
                
                with torch.no_grad():
                    generated_ids = self.model.generate(
                        **model_inputs,
                        max_new_tokens=256,
                        temperature=0.6,
                        do_sample=True
                    )
                
                # Slicing input prompt
                generated_ids = [
                    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
                ]
                response_text = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
                
                # Check if output is valid JSON
                json_start = response_text.find('{')
                json_end = response_text.rfind('}') + 1
                if json_start != -1 and json_end != -1:
                    json_str = response_text[json_start:json_end]
                    json.loads(json_str) # test validation
                    return json_str
            except Exception as e:
                print(f"[AICopilot] Inference error: {e}. Executing fallback recommendation engine.")
                
        # High fidelity fallback rules (mimicking LLM output structure)
        # Determine risks
        if distance > 1500 or traffic > 0.7 or weather > 0.7:
            delay_risk = "High"
        elif distance > 700 or traffic > 0.4 or weather > 0.4:
            delay_risk = "Medium"
        else:
            delay_risk = "Low"
            
        if safety_score < 70:
            compliance_risk = "High"
        elif safety_score < 85:
            compliance_risk = "Medium"
        else:
            compliance_risk = "Low"
            
        est_price = (distance * 1.5) + (weight * 0.05)
        price_status = "High" if est_price > 3000 else ("Fair" if est_price > 1000 else "Low")
        
        rec_text = (
            f"The shipment spanning {distance:.0f} miles is approved with {delay_risk} transit risk and {compliance_risk} carrier safety compliance. "
            f"Carrier is selected based on a safety profile of {safety_score:.0f}/100."
        )
        
        res = {
            "price_status": price_status,
            "delay_risk": delay_risk,
            "compliance_risk": compliance_risk,
            "recommendation_text": rec_text
        }
        return json.dumps(res, indent=4)


## 🖥️ Step 6: Write Streamlit Dashboard Application (`app.py`)
Creates a beautiful dashboard offering User login/signup interfaces, forgot password handlers, interactive calculators, Admin Console, and Copilot Chat.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import time
import os
import json
import joblib
import sqlite3
import datetime
from typing import Dict, Tuple, Optional

# Import local engines (written to same directory in Colab)
from db_engine import (
    validate_email, validate_password, check_password_strength,
    create_jwt_token, decode_jwt_token, check_user_lockout,
    handle_failed_login, reset_failed_login, request_otp,
    send_otp_email, verify_otp, DB_FILE
)
from copilot_engine import AICopilot

# Page configuration
st.set_page_config(
    page_title="FreightQuote AI Enterprise",
    page_icon="🚚",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom premium styling using CSS injection
st.markdown("""
<style>
    /* Main Layout Styling */
    .main {
        background: linear-gradient(135deg, #121824 0%, #0c0f17 100%);
        color: #e2e8f0;
    }
    
    /* Headers styling */
    h1, h2, h3 {
        color: #ffffff !important;
        font-family: 'Outfit', 'Inter', sans-serif;
        font-weight: 700;
        letter-spacing: -0.02em;
    }
    
    /* Card Styles */
    .metric-card {
        background: rgba(30, 41, 59, 0.45);
        backdrop-filter: blur(12px);
        border: 1px solid rgba(255, 255, 255, 0.08);
        border-radius: 12px;
        padding: 24px;
        box-shadow: 0 4px 20px rgba(0, 0, 0, 0.2);
        margin-bottom: 20px;
    }
    
    /* Buttons */
    .stButton>button {
        background: linear-gradient(90deg, #3b82f6 0%, #1d4ed8 100%);
        color: white;
        border: none;
        border-radius: 8px;
        font-weight: 600;
        padding: 10px 24px;
        transition: all 0.3s ease;
    }
    .stButton>button:hover {
        background: linear-gradient(90deg, #60a5fa 0%, #2563eb 100%);
        transform: translateY(-2px);
        box-shadow: 0 4px 15px rgba(59, 130, 246, 0.4);
    }
    
    /* Inputs */
    .stTextInput>div>div>input {
        background-color: #1e293b !important;
        border: 1px solid #475569 !important;
        color: #f8fafc !important;
        border-radius: 8px !important;
    }
    
    /* Sidebar styling */
    section[data-testid="stSidebar"] {
        background-color: #0f172a !important;
        border-right: 1px solid rgba(255, 255, 255, 0.05);
    }
</style>
""", unsafe_allow_html=True)

# Helper function to get SQLite connection
def get_db_connection():
    conn = sqlite3.connect(DB_FILE)
    conn.row_factory = sqlite3.Row
    return conn

# Helper function to check if the user is an admin
def is_admin(user_email: str) -> bool:
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT role FROM users WHERE email = ?", (user_email,))
    row = cursor.fetchone()
    conn.close()
    return row and row['role'] == 'Admin'

# Session state initialization
if 'token' not in st.session_state:
    st.session_state.token = None
if 'user_email' not in st.session_state:
    st.session_state.user_email = None
if 'user_role' not in st.session_state:
    st.session_state.user_role = None
if 'auth_checked' not in st.session_state:
    st.session_state.auth_checked = False
if 'view' not in st.session_state:
    st.session_state.view = "login"  # login, signup, forgot_pw_question, forgot_pw_otp
if 'copilot_instance' not in st.session_state:
    # Initialize the copilot once to reuse
    st.session_state.copilot_instance = AICopilot()
    st.session_state.copilot_instance.load_model()

# Decode JWT token to maintain session
if st.session_state.token and not st.session_state.auth_checked:
    payload = decode_jwt_token(st.session_state.token)
    if payload:
        st.session_state.user_email = payload['email']
        st.session_state.user_role = payload['role']
        st.session_state.auth_checked = True
    else:
        st.session_state.token = None
        st.session_state.user_email = None
        st.session_state.user_role = None

# LOGOUT HANDLER
def handle_logout():
    st.session_state.token = None
    st.session_state.user_email = None
    st.session_state.user_role = None
    st.session_state.auth_checked = False
    st.session_state.view = "login"
    st.rerun()

# ----------------- VIEWS -----------------

# SIGNUP VIEW
def render_signup():
    st.markdown("<h1>🚚 Create a New Account</h1>", unsafe_allow_html=True)
    st.markdown("<p style='color: #94a3b8;'>Access FreightQuote AI enterprise analytics suites.</p>", unsafe_allow_html=True)
    
    with st.form("signup_form"):
        email = st.text_input("Corporate Email Address")
        password = st.text_input("Password", type="password")
        confirm_password = st.text_input("Confirm Password", type="password")
        
        # Security Question Setup
        question = st.selectbox(
            "Security Question (used for account recovery)",
            [
                "What is your mother's maiden name?",
                "What was the name of your first pet?",
                "What is your favorite color?",
                "In what city were you born?"
            ]
        )
        answer = st.text_input("Security Answer")
        
        # Real-time Password Strength meter inside the form (as description)
        strength = check_password_strength(password) if password else "Enter password"
        color = "#ef4444" if strength == "Weak" else ("#eab308" if strength == "Average" else "#22c55e")
        st.markdown(f"Password Strength: <b style='color: {color};'>{strength}</b>", unsafe_allow_html=True)
        
        submit_btn = st.form_submit_button("Sign Up")
        
        if submit_btn:
            if not validate_email(email):
                st.error("Invalid email address format.")
            elif not validate_password(password):
                st.error("Password must be at least 8 characters long and contain uppercase, lowercase, numbers, and special characters.")
            elif password != confirm_password:
                st.error("Passwords do not match.")
            elif not answer.strip():
                st.error("Security answer cannot be blank.")
            else:
                conn = get_db_connection()
                cursor = conn.cursor()
                cursor.execute("SELECT id FROM users WHERE email = ?", (email,))
                if cursor.fetchone():
                    st.error("Email is already registered.")
                    conn.close()
                else:
                    hashed = bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
                    cursor.execute("""
                    INSERT INTO users (email, password_hash, security_question, security_answer, role, created_at)
                    VALUES (?, ?, ?, ?, 'User', ?)
                    """, (email, hashed, question, answer.strip(), datetime.datetime.now().isoformat()))
                    conn.commit()
                    conn.close()
                    st.success("Account created successfully! Please log in.")
                    st.session_state.view = "login"
                    time.sleep(1.5)
                    st.rerun()
                    
    if st.button("Already have an account? Log In"):
        st.session_state.view = "login"
        st.rerun()

# LOGIN VIEW
def render_login():
    st.markdown("<h1>🚚 Log In to FreightQuote AI</h1>", unsafe_allow_html=True)
    st.markdown("<p style='color: #94a3b8;'>Enterprise Logistics Optimization & Smart Carrier Routing</p>", unsafe_allow_html=True)
    
    # Check if there is a local testing helper OTP code
    if os.path.exists("mock_otp.txt"):
        try:
            with open("mock_otp.txt", "r") as f:
                mock_data = f.read().strip()
                if ":" in mock_data:
                    m_email, m_otp = mock_data.split(":", 1)
                    st.info(f"💡 [Colab Helper] Last requested OTP for **{m_email}** is **{m_otp}**")
        except Exception:
            pass

    with st.form("login_form"):
        email = st.text_input("Corporate Email Address")
        password = st.text_input("Password", type="password")
        submit_btn = st.form_submit_button("Log In")
        
        if submit_btn:
            if not email or not password:
                st.error("Please enter email and password.")
            else:
                # Check Lockout Policy first
                locked, lock_msg = check_user_lockout(email)
                if locked:
                    st.error(lock_msg)
                else:
                    conn = get_db_connection()
                    cursor = conn.cursor()
                    cursor.execute("SELECT password_hash, role, id FROM users WHERE email = ?", (email,))
                    row = cursor.fetchone()
                    conn.close()
                    
                    if row and bcrypt.checkpw(password.encode('utf-8'), row['password_hash'].encode('utf-8')):
                        # Success, reset attempts & logs JWT
                        reset_failed_login(email)
                        token = create_jwt_token(email, row['role'])
                        st.session_state.token = token
                        st.session_state.user_email = email
                        st.session_state.user_role = row['role']
                        st.session_state.auth_checked = True
                        st.success("Login successful!")
                        time.sleep(1.0)
                        st.rerun()
                    else:
                        # Fail attempts
                        fail_msg = handle_failed_login(email)
                        st.error(fail_msg)
                        
    col1, col2, col3 = st.columns(3)
    with col1:
        if st.button("Reset via Security Question"):
            st.session_state.view = "forgot_pw_question"
            st.rerun()
    with col2:
        if st.button("Reset via Gmail OTP"):
            st.session_state.view = "forgot_pw_otp"
            st.rerun()
    with col3:
        if st.button("Create Account (Sign Up)"):
            st.session_state.view = "signup"
            st.rerun()

# FORGOT PASSWORD - SECURITY QUESTION VIEW
def render_forgot_question():
    st.markdown("<h1>🔒 Password Reset via Security Question</h1>", unsafe_allow_html=True)
    
    email = st.text_input("Enter your registered email address")
    
    if email:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT security_question FROM users WHERE email = ?", (email,))
        row = cursor.fetchone()
        conn.close()
        
        if not row:
            st.error("Email address not found in system.")
        else:
            st.info(f"Question: {row['security_question']}")
            
            with st.form("security_question_reset_form"):
                answer = st.text_input("Your Security Answer")
                new_password = st.text_input("New Password", type="password")
                confirm_password = st.text_input("Confirm New Password", type="password")
                submit = st.form_submit_button("Reset Password")
                
                if submit:
                    conn = get_db_connection()
                    cursor = conn.cursor()
                    cursor.execute("SELECT security_answer FROM users WHERE email = ?", (email,))
                    real_answer = cursor.fetchone()['security_answer']
                    
                    if answer.strip().lower() != real_answer.lower():
                        st.error("Incorrect security answer.")
                        conn.close()
                    elif not validate_password(new_password):
                        st.error("Password must be >= 8 characters and contain uppercase, lowercase, numbers, and symbols.")
                        conn.close()
                    elif new_password != confirm_password:
                        st.error("Passwords do not match.")
                        conn.close()
                    else:
                        hashed = bcrypt.hashpw(new_password.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
                        cursor.execute("UPDATE users SET password_hash = ? WHERE email = ?", (hashed, email))
                        conn.commit()
                        conn.close()
                        reset_failed_login(email) # Reset lockout attempts on password change
                        st.success("Password reset successful! Please login.")
                        st.session_state.view = "login"
                        time.sleep(1.5)
                        st.rerun()
                        
    if st.button("Cancel & Go Back"):
        st.session_state.view = "login"
        st.rerun()

# FORGOT PASSWORD - OTP VIEW
def render_forgot_otp():
    st.markdown("<h1>📧 Password Reset via OTP Code</h1>", unsafe_allow_html=True)
    
    email = st.text_input("Enter your registered email address")
    
    if 'otp_requested' not in st.session_state:
        st.session_state.otp_requested = False
        
    if email:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT id FROM users WHERE email = ?", (email,))
        row = cursor.fetchone()
        conn.close()
        
        if not row:
            st.error("Email address not found in system.")
        else:
            if st.button("Request OTP Code"):
                success, msg, code = request_otp(email)
                if success:
                    st.success(msg)
                    # Trigger background email delivery (SMTP or local mock log)
                    send_otp_email(email, code)
                    st.session_state.otp_requested = True
                    time.sleep(1.5)
                    st.rerun()
                else:
                    st.error(msg)
                    
            if st.session_state.otp_requested:
                # Helper for local testing to show OTP in UI
                if os.path.exists("mock_otp.txt"):
                    with open("mock_otp.txt", "r") as f:
                        mock_data = f.read().strip()
                        if ":" in mock_data:
                            m_email, m_otp = mock_data.split(":", 1)
                            if m_email == email:
                                st.info(f"💡 [Colab Helper] OTP verification code is: **{m_otp}**")
                                
                with st.form("otp_reset_form"):
                    code = st.text_input("Enter 6-digit OTP Code")
                    new_password = st.text_input("New Password", type="password")
                    confirm_password = st.text_input("Confirm New Password", type="password")
                    submit = st.form_submit_button("Reset Password")
                    
                    if submit:
                        # Verify OTP
                        valid, otp_msg = verify_otp(email, code)
                        if not valid:
                            st.error(otp_msg)
                        elif not validate_password(new_password):
                            st.error("Password must be >= 8 characters and contain uppercase, lowercase, numbers, and symbols.")
                        elif new_password != confirm_password:
                            st.error("Passwords do not match.")
                        else:
                            # Complete reset
                            conn = get_db_connection()
                            cursor = conn.cursor()
                            hashed = bcrypt.hashpw(new_password.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
                            cursor.execute("UPDATE users SET password_hash = ? WHERE email = ?", (hashed, email))
                            # Delete used OTP
                            cursor.execute("DELETE FROM otps WHERE email = ?", (email,))
                            conn.commit()
                            conn.close()
                            reset_failed_login(email)
                            st.session_state.otp_requested = False
                            st.success("Password reset successful! Please login.")
                            st.session_state.view = "login"
                            time.sleep(1.5)
                            st.rerun()
                            
    if st.button("Cancel & Go Back"):
        st.session_state.otp_requested = False
        st.session_state.view = "login"
        st.rerun()

# USER DASHBOARD TAB - MODEL CARD VIEW
def render_model_card():
    st.markdown("<h2>📊 ML Model Specifications & Card</h2>", unsafe_allow_html=True)
    if os.path.exists("ml_model_card.md"):
        with open("ml_model_card.md", "r") as f:
            st.markdown(f.read())
    else:
        st.warning("Model card file not generated. Run model training pipeline cell in the notebook first.")

# USER DASHBOARD TAB - SHIPMENT CALCULATORS
def render_calculators():
    st.markdown("<h2>⚡ Predictive Shipments Optimizer</h2>", unsafe_allow_html=True)
    st.markdown("<p style='color: #94a3b8;'>Utilizes winner algorithms trained in the ML pipeline to predict pricing, route delay risks, and compliance.</p>", unsafe_allow_html=True)
    
    col1, col2, col3 = st.columns(3)
    
    # File check
    pricing_model_exists = os.path.exists("best_pricing_model.joblib")
    route_model_exists = os.path.exists("best_route_delay_model.joblib")
    compliance_model_exists = os.path.exists("best_carrier_compliance_model.joblib")
    
    if not (pricing_model_exists and route_model_exists and compliance_model_exists):
        st.warning("⚠️ Warning: One or more ML models are missing. Ensure all notebook pipeline cells have run successfully.")
        
    with col1:
        st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
        st.markdown("<h3>💲 Price Estimator (Agent 1)</h3>", unsafe_allow_html=True)
        dist = st.number_input("Distance (miles)", min_value=1.0, value=500.0, step=50.0)
        weight = st.number_input("Weight (lbs)", min_value=10.0, value=15000.0, step=500.0)
        fuel = st.slider("Fuel Index Price ($/gal)", 1.0, 7.0, 3.5, 0.1)
        rating = st.slider("Carrier Quality Rating (1-5)", 1.0, 5.0, 4.2, 0.1)
        
        if st.button("Predict Pricing") and pricing_model_exists:
            scaler = joblib.load("pricing_scaler.joblib")
            model = joblib.load("best_pricing_model.joblib")
            
            features = pd.DataFrame([[dist, weight, fuel, rating]], columns=['distance_miles', 'weight_lbs', 'fuel_price_index', 'carrier_rating'])
            scaled_features = scaler.transform(features)
            predicted_price = model.predict(scaled_features)[0]
            st.markdown(f"<div style='font-size: 24px; font-weight: bold; color: #22c55e;'>Est Price: ${predicted_price:,.2f}</div>", unsafe_allow_html=True)
        st.markdown("</div>", unsafe_allow_html=True)
        
    with col2:
        st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
        st.markdown("<h3>⏰ Route Delay Predictor (Agent 2)</h3>", unsafe_allow_html=True)
        route_dist = st.number_input("Route Distance (miles)", min_value=1.0, value=650.0, step=50.0, key="r_dist")
        traffic = st.slider("Traffic Congestion Level", 0.0, 1.0, 0.4, 0.05)
        weather = st.slider("Extreme Weather Index", 0.0, 1.0, 0.2, 0.05)
        delay_rate = st.slider("Carrier Average Delay Rate", 0.0, 1.0, 0.15, 0.01)
        
        if st.button("Check Delay Risk") and route_model_exists:
            scaler = joblib.load("route_scaler.joblib")
            model = joblib.load("best_route_delay_model.joblib")
            
            features = pd.DataFrame([[route_dist, traffic, weather, delay_rate]], columns=['distance_miles', 'traffic_density', 'weather_severity', 'carrier_delay_rate'])
            scaled_features = scaler.transform(features)
            
            prediction = model.predict(scaled_features)[0]
            prob = model.predict_proba(scaled_features)[0][1] if hasattr(model, "predict_proba") else (0.95 if prediction == 1 else 0.05)
            
            color = "#ef4444" if prediction == 1 else "#22c55e"
            status = "DELAY RISK DETECTED" if prediction == 1 else "ON-TIME TRANSIT PROJECTION"
            st.markdown(f"<div style='font-weight: bold; color: {color};'>{status} ({prob*100:.1f}% probability)</div>", unsafe_allow_html=True)
        st.markdown("</div>", unsafe_allow_html=True)
        
    with col3:
        st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
        st.markdown("<h3>🛡️ Carrier Compliance Check (Agent 3)</h3>", unsafe_allow_html=True)
        safety = st.slider("Safety Audit Score", 0.0, 100.0, 85.0, 1.0)
        ins = st.checkbox("Valid Insurance Status", value=True)
        maint = st.checkbox("Maintenance Inspections Pass", value=True)
        violations = st.number_input("Violations Count (Last 12mo)", min_value=0, value=0, step=1)
        years = st.number_input("Carrier Operating Years", min_value=0.5, value=5.0, step=0.5)
        
        if st.button("Evaluate Carrier Compliance") and compliance_model_exists:
            scaler = joblib.load("compliance_scaler.joblib")
            model = joblib.load("best_carrier_compliance_model.joblib")
            
            ins_val = 1 if ins else 0
            maint_val = 1 if maint else 0
            
            features = pd.DataFrame([[safety, ins_val, maint_val, violations, years]], 
                                    columns=['safety_score', 'insurance_validity', 'maintenance_checks_pass', 'violations_count', 'years_in_service'])
            scaled_features = scaler.transform(features)
            prediction = model.predict(scaled_features)[0]
            
            color = "#22c55e" if prediction == 1 else "#ef4444"
            status = "COMPLIANT / APPROVED" if prediction == 1 else "NON-COMPLIANT / HOLD"
            st.markdown(f"<b style='color: {color};'>{status}</b>", unsafe_allow_html=True)
        st.markdown("</div>", unsafe_allow_html=True)

# USER DASHBOARD TAB - AI COPILOT
def render_copilot():
    st.markdown("<h2>🤖 AI Copilot (Qwen2.5-3B-Instruct)</h2>", unsafe_allow_html=True)
    st.markdown("<p style='color: #94a3b8;'>Ask recommendations based on real-time shipment constraints. Output is parsed in structured JSON format.</p>", unsafe_allow_html=True)
    
    # Check fallback/GPU status
    copilot = st.session_state.copilot_instance
    if copilot.gpu_available and copilot.model is not None:
        st.success("🟢 GPU Mode: Qwen2.5-3B Running in quantized 4-bit mode.")
    else:
        st.info("🔵 Fallback Mode: Running deterministic rules recommendations (CPU mode).")
        
    col1, col2 = st.columns([1, 1])
    
    with col1:
        dist = st.number_input("Distance", min_value=1.0, value=850.0, step=10.0, key="cp_dist")
        weight = st.number_input("Weight", min_value=1.0, value=22000.0, step=100.0, key="cp_w")
        traffic = st.slider("Traffic Level", 0.0, 1.0, 0.5, 0.05, key="cp_traf")
        weather = st.slider("Weather Severity", 0.0, 1.0, 0.3, 0.05, key="cp_wea")
        safety = st.slider("Safety Score", 0.0, 100.0, 90.0, 1.0, key="cp_saf")
        
        generate_btn = st.button("Generate Recommendation")
        
    with col2:
        if generate_btn:
            with st.spinner("Analyzing parameters and generating JSON recommendation..."):
                response_str = copilot.generate_recommendation(dist, weight, traffic, weather, safety)
                
                try:
                    res_json = json.loads(response_str)
                    
                    st.markdown("### Recommendation Summary")
                    st.json(res_json)
                    
                    # Styled presentation
                    price_col = "#22c55e" if res_json.get("price_status") == "Low" else ("#eab308" if res_json.get("price_status") == "Fair" else "#ef4444")
                    delay_col = "#22c55e" if res_json.get("delay_risk") == "Low" else ("#eab308" if res_json.get("delay_risk") == "Medium" else "#ef4444")
                    compliance_col = "#22c55e" if res_json.get("compliance_risk") == "Low" else ("#eab308" if res_json.get("compliance_risk") == "Medium" else "#ef4444")
                    
                    st.markdown(f"""
                    - **Pricing Verdict**: <span style='color: {price_col}; font-weight: bold;'>{res_json.get("price_status")}</span>
                    - **Transit Delay Risk**: <span style='color: {delay_col}; font-weight: bold;'>{res_json.get("delay_risk")}</span>
                    - **Compliance Risk**: <span style='color: {compliance_col}; font-weight: bold;'>{res_json.get("compliance_risk")}</span>
                    
                    **Copilot Assessment**:
                    > {res_json.get("recommendation_text")}
                    """, unsafe_allow_html=True)
                except Exception as e:
                    st.error("Failed to parse Copilot output as JSON.")
                    st.text(response_str)

# ADMIN DASHBOARD
def render_admin_dashboard():
    st.markdown("<h1>🛡️ Admin Administration Console</h1>", unsafe_allow_html=True)
    
    menu = ["View Users", "Add User", "Delete User", "Unlock User", "Assign Roles"]
    choice = st.sidebar.selectbox("Admin Controls", menu)
    
    if choice == "View Users":
        st.markdown("<h3>Registered Users</h3>", unsafe_allow_html=True)
        conn = get_db_connection()
        df = pd.read_sql_query("SELECT id, email, role, is_locked, lock_until, failed_attempts, created_at FROM users", conn)
        conn.close()
        
        # Format timestamps
        df['lock_until'] = df['lock_until'].apply(lambda x: "Permanent" if x == 9999999999 else (datetime.datetime.fromtimestamp(x).strftime('%Y-%m-%d %H:%M:%S') if x > 0 else "None"))
        st.dataframe(df, use_container_width=True)
        
    elif choice == "Add User":
        st.markdown("<h3>Add User Manually</h3>", unsafe_allow_html=True)
        with st.form("admin_add_form"):
            new_email = st.text_input("Corporate Email")
            new_password = st.text_input("Temp Password", type="password")
            role_choice = st.selectbox("Role", ["User", "Admin"])
            question = st.text_input("Security Question", value="What is your favorite color?")
            answer = st.text_input("Security Answer", value="Blue")
            submit = st.form_submit_button("Add User")
            
            if submit:
                if not validate_email(new_email):
                    st.error("Invalid email address format.")
                elif not validate_password(new_password):
                    st.error("Password does not meet validation criteria.")
                else:
                    conn = get_db_connection()
                    cursor = conn.cursor()
                    cursor.execute("SELECT id FROM users WHERE email = ?", (new_email,))
                    if cursor.fetchone():
                        st.error("User email already exists.")
                    else:
                        hashed = bcrypt.hashpw(new_password.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
                        cursor.execute("""
                        INSERT INTO users (email, password_hash, security_question, security_answer, role, created_at)
                        VALUES (?, ?, ?, ?, ?, ?)
                        """, (new_email, hashed, question, answer.strip(), role_choice, datetime.datetime.now().isoformat()))
                        conn.commit()
                        st.success(f"User {new_email} successfully added as {role_choice}!")
                    conn.close()
                    
    elif choice == "Delete User":
        st.markdown("<h3>Remove User Access</h3>", unsafe_allow_html=True)
        email_to_delete = st.text_input("User Email Address to Delete")
        
        if st.button("Delete User Account"):
            if email_to_delete == "admin@freightquote.ai":
                st.error("Cannot delete root Admin account.")
            elif email_to_delete == st.session_state.user_email:
                st.error("You cannot delete your own account.")
            else:
                conn = get_db_connection()
                cursor = conn.cursor()
                cursor.execute("SELECT id FROM users WHERE email = ?", (email_to_delete,))
                if not cursor.fetchone():
                    st.error("User email not found.")
                else:
                    cursor.execute("DELETE FROM users WHERE email = ?", (email_to_delete,))
                    conn.commit()
                    st.success(f"User {email_to_delete} was deleted.")
                conn.close()
                
    elif choice == "Unlock User":
        st.markdown("<h3>Administrative Lockout Release</h3>", unsafe_allow_html=True)
        email_to_unlock = st.text_input("Locked Email Address")
        
        if st.button("Unlock and Reset Login Attempts"):
            conn = get_db_connection()
            cursor = conn.cursor()
            cursor.execute("SELECT id FROM users WHERE email = ?", (email_to_unlock,))
            if not cursor.fetchone():
                st.error("User not found.")
            else:
                cursor.execute("""
                UPDATE users 
                SET failed_attempts = 0, is_locked = 0, lock_until = 0 
                WHERE email = ?
                """, (email_to_unlock,))
                conn.commit()
                st.success(f"User {email_to_unlock} account locks have been administratively released.")
            conn.close()
            
    elif choice == "Assign Roles":
        st.markdown("<h3>Modify Role Clearance</h3>", unsafe_allow_html=True)
        email_to_change = st.text_input("Target Email Address")
        new_role = st.selectbox("Assign New Role", ["User", "Admin"])
        
        if st.button("Apply Role Modification"):
            if email_to_change == "admin@freightquote.ai":
                st.error("Cannot modify root Admin account.")
            else:
                conn = get_db_connection()
                cursor = conn.cursor()
                cursor.execute("SELECT id FROM users WHERE email = ?", (email_to_change,))
                if not cursor.fetchone():
                    st.error("User not found.")
                else:
                    cursor.execute("UPDATE users SET role = ? WHERE email = ?", (new_role, email_to_change))
                    conn.commit()
                    st.success(f"User {email_to_change} has been updated to {new_role}.")
                conn.close()


# ----------------- MAIN APP NAVIGATION -----------------

def main():
    if not st.session_state.token:
        # Show Auth Page depending on view
        if st.session_state.view == "signup":
            render_signup()
        elif st.session_state.view == "forgot_pw_question":
            render_forgot_question()
        elif st.session_state.view == "forgot_pw_otp":
            render_forgot_otp()
        else:
            render_login()
    else:
        # Main Dashboard Layout
        st.sidebar.markdown(f"### Logged in as:")
        st.sidebar.markdown(f"**{st.session_state.user_email}**")
        st.sidebar.markdown(f"Security Clearance: `{st.session_state.user_role}`")
        
        # Navigation
        nav_options = ["User Dashboard", "Admin Console"] if st.session_state.user_role == "Admin" else ["User Dashboard"]
        page = st.sidebar.radio("Navigation Pane", nav_options)
        
        if st.sidebar.button("Logout"):
            handle_logout()
            
        if page == "User Dashboard":
            st.markdown("<h1>🚚 FreightQuote AI Enterprise Dashboard</h1>", unsafe_allow_html=True)
            
            sub_tabs = st.tabs(["⚡ Optimize Shipments", "🤖 AI Copilot Assistant", "📊 ML Model Specs (Model Card)"])
            
            with sub_tabs[0]:
                render_calculators()
                
            with sub_tabs[1]:
                render_copilot()
                
            with sub_tabs[2]:
                render_model_card()
                
        elif page == "Admin Console" and st.session_state.user_role == "Admin":
            render_admin_dashboard()

if __name__ == "__main__":
    main()


## ⚙️ Step 7: Initialize Database & Run ML Training Pipeline
Triggers the creation of SQLite schemas, runs the training pipeline on all 15 models (3 agents x 5 algorithms), selects the winners, and writes the Model Card.

In [ ]:
import db_engine
import ml_engine

# Initialize sqlite
db_engine.init_db()

# Execute training pipeline
ml_engine.run_ml_pipeline()

# Display the generated model card
if os.path.exists("ml_model_card.md"):
    with open("ml_model_card.md", "r") as f:
        print("\n--- MODEL CARD ---\n")
        print(f.read())

## 🚀 Step 8: Deploy Streamlit via Ngrok Tunnel
Connects Streamlit background processes to ngrok, exposing a public URL to access the FreightQuote AI dashboard.

In [ ]:
import subprocess
from pyngrok import ngrok

# Configure ngrok token
try:
    ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        print("✅ Ngrok Auth Token loaded successfully.")
    else:
        print("⚠️ NGROK_AUTH_TOKEN not set in Colab Secrets.")
except Exception as e:
    print(f"⚠️ Error setting ngrok token: {e}")

# Start Streamlit in background
print("Starting Streamlit dashboard server...")
streamlit_proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Open ngrok tunnel to Streamlit port
try:
    public_url = ngrok.connect(8501, proto="http")
    print("\n" + "="*60)
    print(f"🚚 FreightQuote AI Dashboard is Live!")
    print(f"Access URL: {public_url}")
    print("="*60 + "\n")
except Exception as e:
    print(f"❌ Ngrok failed to open tunnel: {e}")
    print("Verify if NGROK_AUTH_TOKEN is correct and not used elsewhere.")

## 📄 Step 9: Generate Deliverables & Housekeeping
Outputs standard dependency text, folder documentation, and verification lists. Ensures notebook is cleanly packaged.

In [ ]:
# Write requirements.txt
requirements = """
streamlit>=1.25.0
pyjwt>=2.8.0
bcrypt>=4.0.1
pyngrok>=6.0.0
scikit-learn>=1.2.0
joblib>=1.3.0
pandas>=1.5.0
numpy>=1.23.0
matplotlib>=3.7.0
seaborn>=0.12.0
transformers>=4.31.0
bitsandbytes>=0.41.0
accelerate>=0.21.0
torch>=2.0.0
"""
with open("requirements.txt", "w") as f:
    f.write(requirements.strip())
print("✅ requirements.txt written to disk.\n")

# Output Checklist
checklist = """
===========================================================
             SCREENSHOTS VERIFICATION CHECKLIST
===========================================================
[ ] 1. Sign Up View (validating requirements, password indicator)
[ ] 2. Login View (including invalid attempts warning banner)
[ ] 3. Locked User Account View (showing exact cooldown timers)
[ ] 4. Reset Password Question View (verifying security question)
[ ] 5. Reset Password OTP View (showing Gmail/mock code input)
[ ] 6. User Dashboard (Calculators section with dynamic pricing results)
[ ] 7. AI Copilot Tab (generating JSON format suggestions)
[ ] 8. ML Model Card Markdown panel output
[ ] 9. Admin Dashboard Console (User list & roles management)
[ ] 10. Colab Secrets Panel setup configurations
[ ] 11. Streamlit Launch and Ngrok Tunnel Public URL logs
"""
print(checklist)

# Directory structure
dir_structure = """
===========================================================
               DEPLOYMENT FOLDER STRUCTURE
===========================================================
/content/
  ├── app.py (Streamlit frontend)
  ├── db_engine.py (SQLite configurations & security utilities)
  ├── ml_engine.py (Machine learning training scripts)
  ├── copilot_engine.py (AI Copilot quantized loading & fallbacks)
  ├── freightquote.db (SQLite database schema)
  ├── best_pricing_model.joblib (Agent 1 Model)
  ├── pricing_scaler.joblib (Agent 1 Scaler)
  ├── best_route_delay_model.joblib (Agent 2 Model)
  ├── route_scaler.joblib (Agent 2 Scaler)
  ├── best_carrier_compliance_model.joblib (Agent 3 Model)
  ├── compliance_scaler.joblib (Agent 3 Scaler)
  ├── ml_model_card.md (Generated model performance logs)
  ├── pricing_metrics.json (Agent 1 comparison statistics)
  ├── route_metrics.json (Agent 2 comparison statistics)
  ├── compliance_metrics.json (Agent 3 comparison statistics)
  ├── mock_otp.txt (Offline testing OTP logging fallback)
  └── requirements.txt (Deployment packages definitions)
"""
print(dir_structure)